# SurgVision 실습 — 아이디어에서 원고까지 (설치 없음)

어떤 수술이든, 내 연구 아이디어를 **설계 → 필요한 건수 → 비식별 → 아노테이션 → 데이터 구성 → 모델 비교 → 학습·평가 → 외부검증 → 원고 초안**까지 이 노트북 하나로 따라갑니다. 아노테이션만 CVAT(웹)에서 하고, 나머지는 모두 여기서 합니다.

**사용법:** 위에서부터 셀을 하나씩 실행합니다(▶ 또는 Shift+Enter). `Run all`은 쓰지 않습니다. 각 셀 오른쪽의 입력 칸(폼)을 바꾼 뒤 실행하면 됩니다.

**데이터 원칙:** 이 노트북은 Google 서버(외부 클라우드)에서 돌아갑니다. **예제·공개 데이터**로 연습하거나, IRB가 비식별 프레임의 클라우드 사용을 허용한 경우에만 내 영상을 올리세요. 허용되지 않았으면 원내 컴퓨터용 SurgVision Studio를 씁니다(안내서 부록).

결과물은 런타임이 끝나면 사라집니다. 마지막 11단계에서 결과 ZIP을 꼭 내려받으세요.

In [ ]:
#@title 0. 시작 — 도구와 예제 가져오기
import os, sys, json, subprocess, shutil, glob, pathlib, time
IN_COLAB = 'google.colab' in sys.modules
REPO = pathlib.Path(os.environ.get('SV_REPO', '/content/surgvision-guide'))
if IN_COLAB and not (REPO / 'toolkit').exists():
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/jmleetpl/surgvision-guide', str(REPO)], check=True)
TK = REPO / 'toolkit'
sys.path.insert(0, str(REPO / 'studio')); sys.path.insert(0, str(TK))
import core as C                      # 설계 추천 규칙 (Studio와 같은 코드)
P = pathlib.Path(os.environ.get('SV_WORK', '/content/sv_project')); P.mkdir(parents=True, exist_ok=True)
EX = REPO / 'examples'

def run(script, *args):
    # 툴킷 스크립트를 실행하고 출력을 보여 줍니다. 실패하면 마지막 오류 줄을 남기고 멈춥니다.
    r = subprocess.run([sys.executable, str(TK / script)] + [str(a) for a in args], cwd=P, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    print('\n'.join(l for l in out.splitlines() if 'Warning' not in l and 'warn(' not in l)[-3000:])
    if r.returncode:
        raise SystemExit(f'[실패] {script} — 위 마지막 줄을 확인하세요.')

def upload(local_path=''):
    # Colab: 업로드 창. 로컬 테스트: 주어진 경로.
    if IN_COLAB:
        from google.colab import files
        got = files.upload()
        return [str(P / n) for n in got]  # Colab saves to cwd
    return [local_path] if local_path else []

def download(path):
    if IN_COLAB:
        from google.colab import files
        files.download(str(path))
    else:
        print('저장됨:', path)

if IN_COLAB:
    os.chdir(P)
import torch
print('작업 폴더:', P, '| 툴킷:', TK.exists(), '| 예제:', EX.exists())
print('장치:', 'GPU ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (예제는 CPU로 충분)')
print('ffmpeg:', shutil.which('ffmpeg') is not None)

## 1. 아이디어 → 설계
질문을 모델이 풀 수 있는 모양으로 바꿉니다. **어디에 있나** → 분할(seg) · **이 장면은 어떤 상태인가** → 프레임 분류(tag) · **언제부터 언제까지** → phase 구간(interval).
클래스 이름은 영문 소문자이며 CVAT 라벨과 글자까지 같아야 합니다. 경계 규칙은 칠할 때 망설이는 지점을 3개 이상 적습니다. 예제는 아래 기본값 그대로 실행하세요.

In [ ]:
#@title 1. 설계 저장
제목 = "예제: 복강경 담낭절제 간 표면 분할"  #@param {type:"string"}
영문제목 = "Liver surface segmentation in laparoscopic cholecystectomy video"  #@param {type:"string"}
수술 = "laparoscopic cholecystectomy"  #@param {type:"string"}
영상종류 = "laparoscopic"  #@param ["laparoscopic", "robotic", "endoscopic", "microscopic", "open"]
연구질문 = "모델이 화면에 보이는 간 표면을 환자 단위로 분리된 시험 환자에서 얼마나 정확히 분할하는가?"  #@param {type:"string"}
과제 = "seg"  #@param ["seg", "tag", "interval"]
클래스 = "liver"  #@param {type:"string"}
정의 = "liver: 화면에 실제로 보이는 간 표면. 가려진 부분은 추정하지 않는다."  #@param {type:"string"}
경계규칙 = "담낭과 맞닿은 경계는 장막선에서 자른다; 기구가 가리면 기구는 칠하지 않는다; 반사광은 간 위면 간으로 칠한다"  #@param {type:"string"}
대상없는프레임흔함 = True  #@param {type:"boolean"}
작은구조 = False  #@param {type:"boolean"}
드물지만중요한클래스 = False  #@param {type:"boolean"}
데이터 = "public"  #@param ["public", "own_video"]
계획환자수 = 5  #@param {type:"integer"}
초당프레임 = 1.0  #@param {type:"number"}
영상당아노테이션장수 = 6  #@param {type:"integer"}

cls = [c.strip() for c in 클래스.split(",") if c.strip()]
defs = dict(x.split(":", 1) for x in 정의.split(";") if ":" in x)
defs = {k.strip(): v.strip() for k, v in defs.items()}
hard = [h.strip() for h in 경계규칙.split(";") if h.strip()]
assert all(c in defs for c in cls), f"정의가 없는 클래스가 있습니다: {[c for c in cls if c not in defs]}"
assert len(hard) >= 3, "경계 규칙을 ; 로 구분해 3개 이상 적어 주세요."
pack = C.new_pack(slug=C.slugify(영문제목), title_ko=제목, title_en=영문제목, question=연구질문, rationale="",
                  task_type="segmentation" if 과제 == "seg" else "classification", mode=과제, classes=cls,
                  definitions=defs, hard_cases=hard, source=데이터, procedure=수술, modality=영상종류,
                  n_patients=계획환자수, fps=초당프레임, frames_per_video=영상당아노테이션장수, min_gap=2, annotators=1,
                  absent_common=대상없는프레임흔함, small_structure=작은구조, rare_important=드물지만중요한클래스)
pack["training"]["epochs"] = 6
C.wj(P / "pack.json", pack); C.write_config(P, pack)
m = pack["metrics"]
print("클래스:", pack["class_names"])
print("1차 지표:", m["primary"]); print("보조 지표:", ", ".join(m["secondary"])); print("이유:", m["rationale"])
print("아노테이션 방법:", C.recommend(pack["task_type"], 과제, False, False, False)["annotation"])
print("8단계 모델 후보:", ", ".join(m["candidates"]))

## 2. 의미 있는 건수
결과의 95% 신뢰구간을 ±얼마로 할지 정하면 필요한 **시험 환자 수**가 나옵니다. 같은 환자의 프레임은 서로 비슷해서(ICC) 환자를 늘리는 것이 프레임을 늘리는 것보다 효과적입니다. 학습에 필요한 환자 수는 9단계 학습곡선으로 추정합니다.

In [ ]:
#@title 2. 필요한 시험 환자 수
예상성능 = 0.80  #@param {type:"number"}
프레임간표준편차 = 0.15  #@param {type:"number"}
환자내상관ICC = 0.3  #@param {type:"number"}
환자당시험프레임 = 20  #@param {type:"integer"}
신뢰구간반폭 = 0.05  #@param {type:"number"}
args = ["precision", "--metric", "dice" if pack["task_type"] == "segmentation" else "proportion", "--expected", 예상성능,
        "--icc", 환자내상관ICC, "--frames-per-patient", 환자당시험프레임, "--half-width", 신뢰구간반폭, "--out", P / "sample_size"]
if pack["task_type"] == "segmentation":
    args += ["--sd", 프레임간표준편차]
run("sample_size.py", *args)

## 3. 거버넌스
내 병원 영상이면 IRB 승인과 **클라우드 사용 허용**이 문서로 확인돼야 이 노트북을 쓸 수 있습니다. 확실하지 않으면 '아니오'로 두세요. 그 경우 이 노트북은 4단계에서 멈추고, 원내용 Studio를 쓰도록 안내합니다.

In [ ]:
#@title 3. 거버넌스 결정
IRB상태 = "public_data"  #@param ["public_data", "approved", "exempt_confirmed", "pending"]
클라우드사용허용 = True  #@param {type:"boolean"}
결정자이니셜 = "me"  #@param {type:"string"}
gov = {"irb_status": IRB상태, "compute_platform": "cloud_approved" if 클라우드사용허용 else "local_only",
       "annotation_platform": "cvat_cloud" if 클라우드사용허용 else "cvat_selfhosted",
       "data_class": "public_dataset" if 데이터 == "public" else "deidentified_frames",
       "decided_by": 결정자이니셜, "decided_on": time.strftime("%Y-%m-%d")}
C.wj(P / "governance_decision.json", gov)
ok = IRB상태 in ("public_data", "approved", "exempt_confirmed") and 클라우드사용허용
print("결정:", gov); print("이 노트북으로 진행 가능" if ok else "[STOP] 이 데이터는 Colab에서 처리하지 않습니다. 안내서 부록의 SurgVision Studio(원내 컴퓨터)를 쓰세요.")

## 4. 데이터 넣기
**예제**: 공개 CholecSeg8k로 만든 모의 환자 영상 5개를 씁니다. **내 영상**: 영상들과 `linkage.csv`(original_file,patient_id,video_id)를 하나의 ZIP으로 묶어 올립니다. 파일명은 연구번호로 바뀌고 메타데이터가 지워집니다. 화면에 이름·날짜가 박혀 있으면 가릴 영역 `x,y,w,h`를 적습니다.

In [ ]:
#@title 4. 프레임 추출과 비식별
사용할데이터 = "예제"  #@param ["예제", "내 영상 ZIP 올리기"]
화면글자가리기 = ""  #@param {type:"string"}
LOCAL_VIDEO_ZIP = ""
assert ok, "3단계 결정상 진행할 수 없습니다."
vdir = P / "raw_videos"
if 사용할데이터 == "예제":
    vdir = EX / "example_videos"
else:
    import zipfile
    z = upload(LOCAL_VIDEO_ZIP)[0]
    shutil.rmtree(P / "raw_videos", ignore_errors=True)
    zipfile.ZipFile(z).extractall(P / "raw_videos")
    found = list((P / "raw_videos").rglob("linkage.csv"))
    assert found, "ZIP 안에 linkage.csv가 없습니다."
    vdir = found[0].parent
args = ["--videos", vdir, "--linkage", vdir / "linkage.csv", "--out", P / "frames", "--fps", 초당프레임, "--modality", 영상종류]
if 화면글자가리기:
    args += ["--blackout", 화면글자가리기]
run("extract_frames.py", *args)

## 5. 화면 검토 — 체내 밖·식별 가능 장면 제외
아래 접촉 시트를 모두 봅니다. 수술실·얼굴·모니터·서류·카메라 삽입 전후 장면의 프레임 이름(그림 아래 글자, 예: `V01_000012`)을 `제외할프레임`에 쉼표로 적습니다. 빨간 `OOB?`는 색으로 고른 후보일 뿐입니다. 예제는 비워 두면 됩니다.

In [ ]:
#@title 5-1. 접촉 시트 보기
from IPython.display import Image as IImage, display
for p_ in sorted((P / "frames" / "review").glob("contact_*.jpg")):
    print(p_.name); display(IImage(str(p_)))

In [ ]:
#@title 5-2. 검토 결과 저장
제외할프레임 = ""  #@param {type:"string"}
후보표시취소 = ""  #@param {type:"string"}
검토자 = "me"  #@param {type:"string"}
rows = C.read_csv(P / "frames" / "frames_manifest.csv")
ex_ = {s.strip() for s in 제외할프레임.split(",") if s.strip()}; keep_ = {s.strip() for s in 후보표시취소.split(",") if s.strip()}
for r in rows:
    if r["sample_id"] in ex_: r["oob_flag"] = "1"
    if r["sample_id"] in keep_: r["oob_flag"] = "0"
    r["oob_reviewed"], r["oob_reviewer"] = "1", 검토자
C.write_csv(rows, P / "frames" / "frames_manifest.csv")
print("제외", sum(r["oob_flag"] == "1" for r in rows), "/", len(rows), "프레임 — 검토 완료")

## 6. 아노테이션 (CVAT)
1. 아래 6-1을 실행해 **영상별 배치 ZIP**을 받습니다(예제는 `V01` 하나만 받아도 됩니다).
2. [app.cvat.ai](https://app.cvat.ai/) → Tasks → **+** → Create new task. 이름 `<프로젝트>_<영상번호>`, 라벨은 1단계 클래스 이름 그대로, 파일은 배치 ZIP 하나.
3. 분할: **Brush** 또는 **Polygon**으로 칠하기 → **Save mask** → 상단 **Save**. 분류(tag): 프레임마다 **Tag** 하나. **보간·Track·자동 추적은 쓰지 않습니다**(떨어진 프레임이라 아무도 안 그린 라벨이 생김).
4. Actions → **Export task dataset** → 분할은 *Segmentation Mask 1.1*, 태그는 *CVAT for images 1.1*, Save images 끔.
5. 6-2에서 export ZIP을 올립니다. 예제는 **공개 참조 라벨**을 먼저 자동으로 넣고, 내 export가 같은 프레임을 덮어씁니다. phase 구간(interval)은 `video_id,start_sec,end_sec,label` CSV를 올립니다.

In [ ]:
#@title 6-1. CVAT용 배치 ZIP 만들기·내려받기
배치를만들영상 = "V01"  #@param {type:"string"}
if pack["annotation"]["mode"] != "interval":
    run("select_frames.py", "--frames", P / "frames", "--per-video", 영상당아노테이션장수, "--min-gap-sec", 2, "--out", P / "annotate")
    for v in [s.strip() for s in 배치를만들영상.split(",") if s.strip()]:
        z = P / "annotate" / f"cvat_batch_{v}.zip"
        print("내려받기:", z.name); download(z)
else:
    print("phase 구간 과제는 배치가 필요 없습니다. 영상을 보며 구간 CSV를 만드세요.")

In [ ]:
#@title 6-2. 아노테이션 가져오기
예제참조라벨먼저넣기 = True  #@param {type:"boolean"}
내라벨_출처이름 = "me"  #@param {type:"string"}
LOCAL_EXPORT = ""
mode = {"seg": "seg", "tag": "tag", "interval": "interval"}[pack["annotation"]["mode"]]
fm, lab_dir = P / "frames" / "frames_manifest.csv", P / "labeled"
def imp(export, tag, out):
    base = out / "labeled_manifest.csv"
    args = [mode, "--manifest", base if base.exists() else fm, "--config", P / "config.json", "--out", out, "--source-tag", tag, "--frames", P / "frames"]
    args += (["--intervals", export] if mode == "interval" else ["--export", export])
    run("cvat_import.py", *args)
if 예제참조라벨먼저넣기 and 사용할데이터 == "예제" and mode == "seg":
    for out in (lab_dir, P / "by_source" / "public_reference"):
        imp(EX / "example_reference_labels.zip", "public_reference", out)
print("내 CVAT export(또는 구간 CSV)를 고르세요. 건너뛰려면 업로드 창에서 취소.")
for f in upload(LOCAL_EXPORT):
    for out in (lab_dir, P / "by_source" / 내라벨_출처이름):
        imp(f, 내라벨_출처이름, out)

In [ ]:
#@title 6-3. 판독자 간 일치도
srcs = sorted(d.name for d in (P / "by_source").iterdir()) if (P / "by_source").exists() else []
print("라벨 출처:", srcs)
if len(srcs) >= 2:
    a_ = "public_reference" if "public_reference" in srcs else srcs[0]; b_ = [s for s in srcs if s != a_][0]
    run("compare_labels.py", "--a", P / "by_source" / a_ / "labeled_manifest.csv", "--b", P / "by_source" / b_ / "labeled_manifest.csv",
        "--config", P / "config.json", "--out", P / "agreement", "--name-a", a_, "--name-b", b_)
    print("→ 낮은 프레임을 보며 라벨 정의서를 다듬을 점이 있는지 판단하세요.")
else:
    print("출처가 2개 이상일 때 계산합니다(예: 공개 참조 vs 내 라벨, 판독자 1 vs 2).")

## 7. 데이터 구성
환자 단위로 학습·검증·시험을 나눕니다(같은 환자가 두 곳에 있으면 멈춤). **외부검증 연습**: 예제에서 `외부검증환자`에 `P05`를 적으면 그 환자를 '다른 병원 데이터'로 떼어 두고 10단계에서 평가합니다. 시간적 검증을 원하면 `고정시험환자`에 나중 수술일 환자를 적습니다.

In [ ]:
#@title 7. 환자 단위 분할과 점검
비율 = "0.6,0.2,0.2"  #@param {type:"string"}
고정시험환자 = ""  #@param {type:"string"}
외부검증환자 = "P05"  #@param {type:"string"}
rows = C.read_csv(lab_dir / "labeled_manifest.csv")
ext_p = {s.strip() for s in 외부검증환자.split(",") if s.strip()}
ext_rows = [dict(r, split="external") for r in rows if r["patient_id"] in ext_p and r.get("annotation_status") == "done"]
for r in rows:
    if r["patient_id"] in ext_p and r.get("annotation_status") == "done":
        r["annotation_status"] = "held_out_external"
C.write_csv(rows, P / "internal_manifest.csv")
if ext_rows:
    C.write_csv(ext_rows, P / "external" / "external_manifest.csv"); print("외부검증용으로 떼어 둔 프레임:", len(ext_rows))
args = ["--manifest", P / "internal_manifest.csv", "--ratios", 비율, "--seed", 42, "--out", P / "split"]
if 고정시험환자.strip():
    args += ["--test-patients", 고정시험환자.replace(" ", "")]
run("make_splits.py", *args)
run("data_audit.py", "--manifest", P / "split" / "split_manifest.csv", "--config", P / "config.json", "--out", P / "audit")

## 8. 모델 후보 비교 (검증 자료만)
같은 조건으로 짧게 학습해 **검증 점수**로 비교합니다. 시험 자료는 열지 않습니다. 차이가 작으면 더 가벼운 모델을 고릅니다. CPU에서는 후보당 1~3분 정도 걸립니다(GPU면 더 빠름: 런타임 → 런타임 유형 변경).

In [ ]:
#@title 8. 모델 후보 비교
후보 = ",".join(pack["metrics"]["candidates"])  # 바꾸려면 아래 줄 주석을 풀고 수정
# 후보 = "unet_resnet18,lraspp_mobilenet"
후보당epoch = 2  #@param {type:"integer"}
run("model_search.py", "--config", P / "config.json", "--manifest", P / "split" / "split_manifest.csv", "--out", P / "search",
    "--candidates", 후보, "--epochs", 후보당epoch)
sel = json.load(open(P / "search" / "search_summary.json"))
pack["training"]["arch"] = sel["chosen"]; C.wj(P / "pack.json", pack); C.write_config(P, pack)
print("선택:", sel["chosen"], "—", sel["chosen_reason"])

## 9. 학습·평가
검증으로 가장 좋은 epoch를 고른 뒤 **시험 자료로 한 번** 평가합니다. 결과를 보고 설정을 바꿔 다시 돌렸다면 원고에 그 사실을 적어야 합니다. `weights_changed`와 `checkpoint_reload_verified`가 True인지 먼저 보세요. † 표시는 시험 환자가 5명 미만이라 신뢰구간을 믿을 수 없다는 뜻입니다.

In [ ]:
#@title 9-1. 학습과 시험 평가
epoch = 6  #@param {type:"integer"}
pack["training"]["epochs"] = epoch; C.wj(P / "pack.json", pack); C.write_config(P, pack)
RUN = P / "runs" / time.strftime("%Y%m%d_%H%M%S")
run("train.py", "--config", P / "config.json", "--manifest", P / "split" / "split_manifest.csv", "--out", RUN)
run("make_report.py", "--split", P / "split" / "split_summary.json", "--run", RUN, "--out", RUN / "results")
from IPython.display import Image as IImage, display, Markdown
display(Markdown(open(RUN / "results" / "table2_performance.md").read()))
for f in ("fig_learning_curve.png", "fig_per_case.png"):
    if (RUN / "results" / f).exists(): display(IImage(str(RUN / "results" / f)))
for f in sorted((RUN / "examples").glob("*.png")):
    print(f.name, "(원본 | 정답 | 예측)"); display(IImage(str(f)))

In [ ]:
#@title 9-2. 영상에 적용해 보기 (선택)
영상에적용 = True  #@param {type:"boolean"}
if 영상에적용:
    vids_ = sorted(vdir.glob("*.mp4"))
    test_p = json.load(open(P / "split" / "split_summary.json"))["patients"].get("test", [])
    link_ = {r["patient_id"]: r["original_file"] for r in C.read_csv(vdir / "linkage.csv")}
    v_ = vdir / link_[test_p[0]] if test_p and test_p[0] in link_ else vids_[0]
    print("적용 영상:", v_.name, "(시험 환자 영상이면 시험 결과와 같은 근거이지 추가 평가가 아닙니다)")
    run("infer_video.py", "--checkpoint", RUN / "checkpoint_best.pt", "--video", v_, "--fps", 2, "--out", RUN / "video")
    if (RUN / "video" / "timeline.png").exists(): display(IImage(str(RUN / "video" / "timeline.png")))
    if (RUN / "video" / "overlay.mp4").exists():
        from IPython.display import Video
        display(Video(str(RUN / "video" / "overlay.mp4"), embed=True, width=480))

In [ ]:
#@title 9-3. 학습곡선 — 환자를 더 모으면? (선택)
학습곡선실행 = False  #@param {type:"boolean"}
if 학습곡선실행:
    run("sample_size.py", "curve", "--config", P / "config.json", "--manifest", P / "split" / "split_manifest.csv",
        "--out", P / "sample_size", "--reps", 1, "--epochs", 2)
    if (P / "sample_size" / "learning_curve.png").exists(): display(IImage(str(P / "sample_size" / "learning_curve.png")))
else:
    print("학습 환자 수를 늘려 가며 다시 학습해 '몇 명 더 모으면 좋아지나'를 추정합니다. 켜려면 체크하세요(시간이 더 걸림).")

## 10. 외부검증
학습·모델 선택에 전혀 쓰지 않은 **다른 기관·시기·장비**의 환자에 *고정된* 모델을 적용합니다. 재학습이나 조정은 하지 않습니다. 7단계에서 떼어 둔 환자가 여기에 쓰입니다.

In [ ]:
#@title 10. 외부검증
ext_m = P / "external" / "external_manifest.csv"
if ext_m.exists():
    run("evaluate.py", "--checkpoint", RUN / "checkpoint_best.pt", "--manifest", ext_m, "--name", "external_holdout",
        "--out", P / "external" / "external_holdout", "--internal", RUN / "metrics.json")
else:
    print("외부검증 데이터가 없습니다. 원고 한계에 '외부검증 없음'으로 기록됩니다.")

## 11. 결과·원고 초안
모든 숫자 옆에 `[출처 파일: 키]`가 붙은 초안을 만듭니다. Introduction·Discussion은 개요만 있고, 참고문헌은 PubMed에서 확인한 것만 넣습니다. **결과 ZIP을 꼭 내려받으세요** — 런타임이 끝나면 사라집니다.

In [ ]:
#@title 11. 원고 초안과 결과 내려받기
args = ["--project", P, "--run", RUN, "--out", P / "manuscript_draft.md", "--audit", P / "audit", "--search", P / "search"]
if (P / "sample_size" / "sample_size_precision.json").exists(): args += ["--sample-size", P / "sample_size"]
if (P / "external" / "external_holdout" / "external_metrics.json").exists(): args += ["--external", P / "external" / "external_holdout"]
run("draft_manuscript.py", *args)
display(Markdown(open(P / "manuscript_draft.md").read()))
import zipfile
zp = pathlib.Path("/content" if IN_COLAB else P.parent) / f"surgvision_results_{time.strftime('%Y%m%d_%H%M')}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    skip_dirs = {"raw_videos", "frames", "annotate", "masks"}          # images/masks stay out; only results
    for f in P.rglob("*"):
        rel = f.relative_to(P)
        if not f.is_file() or skip_dirs & set(rel.parts) or f.name == "linkage.csv":
            continue
        if f.suffix == ".pt" and f.parent != RUN:                          # keep only the final model
            continue
        if f.suffix in (".json", ".md", ".csv", ".png", ".mp4", ".pt"):
            z.write(f, rel)
print("결과 ZIP:", zp.name, f"({zp.stat().st_size / 1e6:.1f} MB, 원본 영상·대응표 제외)")
download(zp)